# 多页面网站实践

学习目标：能制作相互连接的目录页、内容页和表单页，并检查资源路径、提交字段与键盘操作。

前置知识：HTML 文档结构、语义元素、相对链接、图片、表格和表单控件；会保存文件并打开浏览器开发者工具。

适用范围：WHATWG HTML Living Standard；使用现代浏览器。表单回显使用 Python 3.12 标准库服务，不需要 JavaScript 或构建工具。

环境准备：[环境配置与运行](README.md)。

工作目录：content/Web与应用开发/html；在浏览器运行配套页面。

本章制作虚构的“周末读书会”小站，配套文字与插图供本机练习。页面不加载外部资源；表单只预览收到的数据，不保存报名、不发送邮件。请用虚构称呼和问题操作。

配套脚本：位于 scripts/14-multi-page-site/。

1. [site/index.html](scripts/14-multi-page-site/site/index.html)：阅读目录与活动安排。
2. [site/notes/reading.html](scripts/14-multi-page-site/site/notes/reading.html)：内容页、子目录链接和插图。
3. [site/join.html](scripts/14-multi-page-site/site/join.html)：参与意向表单。
4. [site/assets/reading.svg](scripts/14-multi-page-site/site/assets/reading.svg)、[site/assets/styles.css](scripts/14-multi-page-site/site/assets/styles.css)：阅读准备图和共用样式。
5. [preview_server.py](scripts/14-multi-page-site/preview_server.py)：提供 site 内的静态资源及 /preview GET 回显。

## 打开配套网站

Step 1：在已激活 Python 环境的终端中，从项目根目录进入本技术目录。

```bash
cd content/Web与应用开发/html
```

Step 2：启动本章预览服务。

```bash
python scripts/14-multi-page-site/preview_server.py
```

Step 3：打开[阅读目录](http://127.0.0.1:8014/index.html)。

服务只监听本机 127.0.0.1:8014

静态服务根目录固定为 content/Web与应用开发/html/scripts/14-multi-page-site/site，因此页面地址不重复包含 scripts 或 site。Notebook 的文件链接仍相对于 content/Web与应用开发/html。

表单必须通过这个服务打开：普通静态文件服务不会自动处理 /preview。端口占用时先检查自己已启动的本章服务，不要重复启动或结束其他进程。

Step 4：在服务终端按 Ctrl+C 停止服务。

## 1 按阅读任务组织页面

读者从目录选择文章，读完后填写参与意向。三个页面各自保留页面标题与主要内容，并用一致的导航连接：

- site/index.html：选择阅读内容、查看活动安排。
- site/notes/reading.html：阅读完整记录，返回目录或进入表单。
- site/join.html：选择场次与主题，进入本机回显页。
- site/assets/：页面共用的图片与样式。

以上是相对本章配套目录的文件位置。HTML 中没有设置 &lt;base&gt;，相对 URL 以当前文档 URL 为基准。目录页进入内容页只需：

```html
<a href="notes/reading.html">给一次阅读留三个位置</a>
<!-- 点击后进入 /notes/reading.html；页面地址与 Notebook 的文件链接写法不同。 -->
```

配套文件：[scripts/14-multi-page-site/site/index.html](scripts/14-multi-page-site/site/index.html) · [浏览器预览](http://127.0.0.1:8014/index.html)

## 2 保持一致的站点导航

本例用 &lt;header&gt; 放站点介绍，&lt;nav&gt; 放主要导航，&lt;main&gt; 放当前页主要内容，&lt;footer&gt; 放页脚。每页的 &lt;title&gt; 与 &lt;h1&gt; 都说明当前页面主题。

- aria-label="站点导航"：为导航区域提供名称。
- aria-current="page"：标明当前页；同一组导航中只标记一个当前项目。
- href：仍负责目标地址；aria-current 不会代替链接或自动产生视觉样式。

目录页中的导航如下，另外两页按各自位置调整 href 和当前页标记：

```html
<nav aria-label="站点导航">
  <ul>
    <li><a href="index.html" aria-current="page">阅读目录</a></li>
    <li><a href="notes/reading.html">阅读记录</a></li>
    <li><a href="join.html">参与意向</a></li>
  </ul>
</nav>
<!-- 逐一打开三个链接；在各页核对当前页标记和页面标题。 -->
```

配套文件：[scripts/14-multi-page-site/site/index.html](scripts/14-multi-page-site/site/index.html) · [浏览器预览](http://127.0.0.1:8014/index.html)

### 2.1 用片段链接跳过重复导航

&lt;body&gt; 内的首个链接指向主要内容：

```html
<a class="skip-link" href="#main">跳到主要内容</a>
```

配套文件：[scripts/14-multi-page-site/site/index.html](scripts/14-multi-page-site/site/index.html) · [浏览器预览](http://127.0.0.1:8014/index.html)

站点页头和导航之后，主要内容从以下开始标签进入。本例每页只有一个 &lt;main&gt;；tabindex="-1" 允许它取得焦点，同时不增加普通 Tab 停靠点。

```html
<main id="main" tabindex="-1">
  <h1>阅读目录</h1>
<!-- 在页首按 Tab、Enter 激活跳转，焦点应进入 main；下一次 Tab 到“给一次阅读留三个位置”。 -->
```

配套文件：[scripts/14-multi-page-site/site/index.html](scripts/14-multi-page-site/site/index.html) · [浏览器预览](http://127.0.0.1:8014/index.html)

## 3 在内容页组织完整文章

内容页把阅读记录放进 &lt;article&gt;，内部按主题使用带 &lt;h2&gt; 的 &lt;section&gt;。&lt;main&gt; 表示页面的主要内容，&lt;article&gt; 表示可独立阅读的文章，二者可以嵌套。

下面是 &lt;main&gt; 内文章的开头；本站使用一个 &lt;h1&gt; 表达每页中心主题，后续分节用 &lt;h2&gt;：

```html
<article>
  <header>
    <h1>给一次阅读留三个位置</h1>
    <p>记录人：小林 · 原创示例</p>
  </header>
```

配套文件：[scripts/14-multi-page-site/site/notes/reading.html](scripts/14-multi-page-site/site/notes/reading.html) · [浏览器预览](http://127.0.0.1:8014/notes/reading.html)

其中一个分节的标题和段落为：

```html
<h2>笔记中留一个问题</h2>
<p>我在笔记里写下：“如果换一个叙述者，这个故事会怎样？”
   这次讨论就从这个问题开始。</p>
```

配套文件：[scripts/14-multi-page-site/site/notes/reading.html](scripts/14-multi-page-site/site/notes/reading.html) · [浏览器预览](http://127.0.0.1:8014/notes/reading.html)

### 3.1 引入图片与图注

&lt;img&gt; 引入外部图片；&lt;figure&gt; 将图片与 &lt;figcaption&gt; 图注组织在一起。本例 SVG 是配套图片资源，绘图实现不需要写进页面正文。

- src：图片地址；../assets/reading.svg 从内容页所在的 notes/ 返回上一层，再进入 assets/。
- alt：图片在当前上下文中传达的信息，不能只写“图片”，也不由图注代替。
- width、height：声明本例图片的 600 × 200 尺寸；配套样式允许图片按比例缩小。

本例用 alt 描述准备的三样物品，图注说明这是一张准备示意图。

```html
<figure>
  <img src="../assets/reading.svg" width="600" height="200"
       alt="本次准备：一本书、一支笔、一本笔记本">
  <figcaption>本次阅读准备示意图。</figcaption>
</figure>
<!-- 直接打开内容页并刷新，图片应显示书、笔和笔记本；检查 alt 与图注各自表达的信息。 -->
```

配套文件：[scripts/14-multi-page-site/site/notes/reading.html](scripts/14-multi-page-site/site/notes/reading.html) · [浏览器预览](http://127.0.0.1:8014/notes/reading.html)

### 3.2 从子目录返回与跨页定位

以下 URL 都以内容页 /notes/reading.html 为基准：

- ../index.html：回到目录页。
- ../index.html#schedule：打开目录页，并定位 id 为 schedule 的活动安排区域。
- ../join.html：进入表单页。
- ../assets/styles.css：加载共用样式。

片段与文件名都按实际拼写检查；HTML URL 使用正斜杠 /，不要写成 Windows 磁盘路径。

```html
<p><a href="../index.html#schedule">查看活动安排</a></p>
<p><a href="../join.html">填写参与意向</a></p>
<!-- “查看活动安排”应定位到目录页的 schedule 区域；“填写参与意向”应打开 /join.html。 -->
```

配套文件：[scripts/14-multi-page-site/site/notes/reading.html](scripts/14-multi-page-site/site/notes/reading.html) · [浏览器预览](http://127.0.0.1:8014/notes/reading.html)

## 4 用表格比较活动安排

活动安排具有行列对应关系，适合使用数据表格。&lt;caption&gt; 说明表格主题，&lt;th&gt; 标识表头，&lt;td&gt; 保存普通数据；导航和页面布局不使用表格。

scope 是 &lt;th&gt; 的属性：

- col：该表头对应所在列。
- row：该表头对应所在行。

这里保留两场虚构活动，便于同时核对行表头与列表头：

```html
<table>
  <caption>周末两场讨论安排</caption>
  <thead>
    <tr><th scope="col">场次</th><th scope="col">时段</th><th scope="col">讨论内容</th></tr>
  </thead>
  <tbody>
    <tr><th scope="row">周六场</th><td>14:00–14:40</td><td>交流阅读问题</td></tr>
    <tr><th scope="row">周日场</th><td>10:00–10:40</td><td>分享笔记片段</td></tr>
  </tbody>
</table>
<!-- 阅读周六、周日两行时，应能辨认场次、时段和讨论内容之间的对应关系。 -->
```

配套文件：[scripts/14-multi-page-site/site/index.html](scripts/14-multi-page-site/site/index.html) · [浏览器预览](http://127.0.0.1:8014/index.html)

## 5 表单：关联标签并约束输入

表单将填写内容交给本机 /preview 回显。页面先说明用途，再给出 &lt;form&gt; 的开始标签：

```html
<p id="preview-note">仅在本机显示提交内容，不保存报名，也不发送邮件。
   输入会出现在地址栏，请勿填写个人资料。标有“必填”的项目需要完成。</p>
<form action="preview" method="get" aria-describedby="preview-note">
```

配套文件：[scripts/14-multi-page-site/site/join.html](scripts/14-multi-page-site/site/join.html) · [浏览器预览](http://127.0.0.1:8014/join.html)

- action="preview"：提交目标，相对当前 /join.html 解析为 /preview。
- method="get"：把提交字段编码到 URL 查询部分；适用于本例不改变报名状态的预览。
- aria-describedby="preview-note"：把前面的说明文字关联为表单描述，不代替各控件的标签。

仅设置 action 不会创建服务端功能；本章开头启动的 Python 服务负责返回回显页。

### 5.1 称呼：标签、字段名与必填

相关控件放在 &lt;fieldset&gt; 中，用 &lt;legend&gt; 给组命名。本例第一组叫“称呼与场次”，其中称呼控件为：

```html
<label for="nickname">称呼（必填）</label>
<input id="nickname" name="nickname" type="text" required
       maxlength="20" autocomplete="off">
<!-- 不填称呼就提交，浏览器应阻止提交并提示补充；点击标签应聚焦称呼框。 -->
```

配套文件：[scripts/14-multi-page-site/site/join.html](scripts/14-multi-page-site/site/join.html) · [浏览器预览](http://127.0.0.1:8014/join.html)

- for 与 id：关联可见标签和控件。
- name="nickname"：提交字段名；输入内容成为该字段的值。
- required：必填属性，本例要求文本值非空。
- maxlength="20"：限制用户输入的文本长度，按 UTF-16 码元计数，不等同于固定的“20 个可见字符”。
- autocomplete="off"：请求浏览器不要为此字段自动补全。

这些属性帮助浏览器检查输入，不能代替服务端校验。

### 5.2 场次：显示文字与提交值

本例 &lt;select&gt; 是必选的单选下拉框。首个直接子项使用空 value 作为提示，用户需要选择实际场次：

```html
<label for="session">场次（必填）</label>
<select id="session" name="session" required>
  <option value="">请选择场次</option>
  <option value="saturday">周六场 · 14:00</option>
  <option value="sunday">周日场 · 10:00</option>
</select>
<!-- 只填写称呼、保留“请选择场次”后提交，应停在表单页；选周六场后提交值为 saturday。 -->
```

配套文件：[scripts/14-multi-page-site/site/join.html](scripts/14-multi-page-site/site/join.html) · [浏览器预览](http://127.0.0.1:8014/join.html)

- 页面显示“周六场 · 14:00”，提交值为 saturday。
- 页面显示“周日场 · 10:00”，提交值为 sunday。

saturday、sunday 是本例与接收程序约定的值，不是 HTML 预定义关键字。

### 5.3 同名复选框可以提交多个条目

两个主题共用 name="topic"，各有独立 id 和 value。它们可以同时勾选；&lt;fieldset&gt; 和 &lt;legend&gt; 说明这组选择的共同用途。

```html
<fieldset>
  <legend>想讨论的主题（可多选）</legend>
  <p class="choice">
    <input id="topic-reading" name="topic" type="checkbox" value="reading">
    <label for="topic-reading">阅读问题</label>
  </p>
  <p class="choice">
    <input id="topic-notes" name="topic" type="checkbox" value="notes">
    <label for="topic-notes">笔记整理</label>
  </p>
</fieldset>
<!-- 同时勾选两项再提交，应保留 topic=reading 和 topic=notes 两个条目。 -->
```

配套文件：[scripts/14-multi-page-site/site/join.html](scripts/14-multi-page-site/site/join.html) · [浏览器预览](http://127.0.0.1:8014/join.html)

未勾选的复选框不提交；接收程序需要保留同名条目，不能只留下最后一个值。

### 5.4 空值与未提交字段不同

问题是选填内容，&lt;textarea&gt; 有 name，因此留空时仍可以提交 question 的空值：

```html
<label for="question">带来的问题（选填）</label>
<textarea id="question" name="question" rows="3" maxlength="120"></textarea>
```

配套文件：[scripts/14-multi-page-site/site/join.html](scripts/14-multi-page-site/site/join.html) · [浏览器预览](http://127.0.0.1:8014/join.html)

临时备忘没有 name，会议链接被 disabled 禁用，二者都不进入本例的提交数据：

```html
<p>
  <label for="local-draft">临时备忘（不提交）</label>
  <input id="local-draft" type="text">
</p>
<p>
  <label for="meeting-link">会议链接（尚未提供，不提交）</label>
  <input id="meeting-link" name="meeting" type="text"
         value="尚未提供" disabled>
<!-- 填写临时备忘后提交：回显中应没有 local-draft 和 meeting 条目。 -->
</p>
```

配套文件：[scripts/14-multi-page-site/site/join.html](scripts/14-multi-page-site/site/join.html) · [浏览器预览](http://127.0.0.1:8014/join.html)

id 服务于标签、片段和脚本等引用；只设置 id 不会产生表单字段。

## 6 对照 GET 请求与回显

表单末尾明确使用提交按钮。按钮没有 name，因此不会额外产生一个按钮字段：

```html
<button type="submit">预览提交内容</button>
<!-- 填好必填项后提交，地址应进入 /preview，查询字段应与回显表格一致。 -->
```

配套文件：[scripts/14-multi-page-site/site/join.html](scripts/14-multi-page-site/site/join.html) · [浏览器预览](http://127.0.0.1:8014/join.html)

在表单页打开 Network，依次操作：

（1）不输入就提交，再只填写称呼提交。检查应仍停留在表单页，且没有新的 /preview 请求；不要求不同浏览器显示相同的提示文案。

（2）称呼填“小林”，选择周六场，勾选两个主题，问题填“想比较两次笔记”。提交后选择 /preview 请求，在 Payload → Query String Parameters 中检查 nickname、session、两条 topic 和 question。

（3）重新填写，不勾选主题、问题留空。检查没有 topic 条目，question 的空值仍保留。

（4）称呼改为“小林&lt;读者&gt;”后提交。回显应显示完整文字，尖括号内的内容不应变成 HTML 元素。

[preview_server.py](scripts/14-multi-page-site/preview_server.py) 解析 GET 查询，保留重复字段和空值，并把输入转义后放入 HTML 文本位置。回显只表明服务收到数据，不表示报名成功；手工构造的不符合表单约束的请求也会回显。实现细节留在配套源码中。

## 7 共用样式并保留可见焦点

页面通过 &lt;link&gt; 加载同一份样式。目录页与表单页位于服务根目录；内容页位于 notes/，因此它在 &lt;head&gt; 中使用：

```html
<link rel="stylesheet" href="../assets/styles.css">
```

配套文件：[scripts/14-multi-page-site/site/notes/reading.html](scripts/14-multi-page-site/site/notes/reading.html) · [浏览器预览](http://127.0.0.1:8014/notes/reading.html)

样式仅辅助阅读。本例单列排版，导航可换行，不用视觉重排改变内容顺序。图片的 max-width: 100% 限制最大宽度，height: auto 保持比例：

```css
img {
  display: block;
  max-width: 100%;
  height: auto;
}
```

配套文件：[scripts/14-multi-page-site/site/assets/styles.css](scripts/14-multi-page-site/site/assets/styles.css) · [浏览器预览](http://127.0.0.1:8014/notes/reading.html)

共用样式还用 :focus-visible 显示焦点轮廓。当前页的粗体与下划线标记表示所在页面，与临时移动的键盘焦点不同。

## 8 按真实访问路线检查

（1）从目录进入内容页，再进入表单，最后返回活动安排。直接输入内容页地址并刷新，确认子目录页面独立打开时也能加载图片与样式。

（2）在 Network 中核对 HTML、CSS、SVG 的请求 URL 和状态；在 Elements 中继续检查标题层级、标签关联和替代文本。资源返回成功不能代替语义与布局检查。

（3）只用键盘从页首操作：Tab 到跳转链接，Enter 跳到主要内容，继续到正文链接或表单控件；Shift+Tab 反向移动，空格切换复选框，用方向键选择场次，最后提交。焦点应可辨，操作可达，没有焦点陷阱。

（4）在约 360 CSS 像素的窄窗口和宽窗口中重复路线，有条件时换第二种浏览器，并记录名称与版本。控件外观和提示文字可以不同，目标地址、必填约束及提交字段应保持一致。

HTML 校验与更完整的可访问性检查见[可访问性与 HTML 检查](<13-可访问性与 HTML 检查.ipynb>)。这些局部检查不能据以宣称整个网站满足全部可访问性要求。

## 本章小结

- 目录、内容和表单各有职责，共享导航与样式，但保留各自的页面主题。
- 相对 URL 从文档基准出发；文件路径、HTTP 服务根与 Notebook 链接要分清。
- 标签关联、提交字段名和提交值各有职责；同名条目、空值与未提交字段需要分别核对。
- 数据回显不等于业务接受；浏览器约束、网络请求和键盘操作也要分别检查。

## 练习

先将 scripts/14-multi-page-site/site/notes/reading.html 复制为 scripts/14-multi-page-site/site/notes/practice.html，将 scripts/14-multi-page-site/site/join.html 复制为 scripts/14-multi-page-site/site/practice.html。只修改副本，各自的当前页导航链接也要指向副本。

（1）在内容页副本增加“讨论后的补记”分节，并链接到表单副本。检查：新标题从属于本页主标题；从 /notes/practice.html 到达 /practice.html；图片与样式无失效请求。

（2）在表单副本增加“选书建议”复选框，提交值为 book-choice。检查：与另一主题同时勾选时保留两个 topic 条目；取消勾选后不再出现 book-choice。

（3）给临时备忘加 name="memo"，并同步修改标签中的“不提交”说明。分别提交空备忘与“讨论后补充”。检查：两次都有 memo 字段，值分别为空与指定文字；能解释为什么只改 id 无效。

（4）暂时去掉内容页副本图片路径中的 ../，刷新并查看失败请求的完整 URL，再修复。检查：能解释错误路径为何落入 notes/assets；修复后重新确认图片显示与替代文本。

### 提示

跨页链接先确定当前文档所在目录；新复选框复用 topic 字段名，但需要独立 id 和对应的 for。空字段值与没有字段不同。练习后可删除两个副本。

### 参考解析

（1）新分节使用 h2，与既有三个分节同级。位于 notes/practice.html 的链接写 ../practice.html；当前页导航在内容副本中写 practice.html，在表单副本中也写 practice.html，但两者所在目录不同，完整目标并不相同。图片和样式继续使用 ../assets/，不用因为文件改名而增加目录层级。

（2）新控件保留 name="topic"，value="book-choice"，并设置唯一 id 与对应 label。与 reading 一起勾选时产生两条 topic；取消新项后只有原项。不能把 name 改成 book-choice，否则会改变字段分组。

（3）增加 name="memo" 后，可提交文本框即使为空也构造一个 memo 条目；填写后值变为“讨论后补充”。id 只负责文档内标识，不能替代提交字段名。不要为可选字段添加 required，否则题目中的空值实验条件已经改变。

（4）从 /notes/practice.html 解析 assets/reading.svg 得到 /notes/assets/reading.svg，静态目录中没有这个文件，预期 404。恢复 ../ 后得到 /assets/reading.svg，返回图片并显示书、笔、笔记本。错误在相对路径基准，不在 SVG 内容或浏览器缓存。

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| WHATWG HTML | [&lt;article&gt;](https://html.spec.whatwg.org/multipage/sections.html#the-article-element)、[&lt;section&gt;](https://html.spec.whatwg.org/multipage/sections.html#the-section-element)、[&lt;nav&gt;](https://html.spec.whatwg.org/multipage/sections.html#the-nav-element)、[&lt;header&gt;](https://html.spec.whatwg.org/multipage/sections.html#the-header-element)、[&lt;footer&gt;](https://html.spec.whatwg.org/multipage/sections.html#the-footer-element)及 [&lt;main&gt;](https://html.spec.whatwg.org/multipage/grouping-content.html#the-main-element)：页面与文章结构；[&lt;img&gt;](https://html.spec.whatwg.org/multipage/embedded-content.html#the-img-element)与 [&lt;th&gt;](https://html.spec.whatwg.org/multipage/tables.html#the-th-element)：图片和表头；[&lt;label&gt;](https://html.spec.whatwg.org/multipage/forms.html#the-label-element)、[&lt;select&gt;](https://html.spec.whatwg.org/multipage/form-elements.html#the-select-element)、[Form submission](https://html.spec.whatwg.org/multipage/form-control-infrastructure.html#form-submission)与 [Constructing the entry list](https://html.spec.whatwg.org/multipage/form-control-infrastructure.html#constructing-the-entry-list)：控件关联、必选提示项、GET、重复字段及未提交条件；[autocomplete](https://html.spec.whatwg.org/multipage/form-control-infrastructure.html#attr-fe-autocomplete)的 off 值。 |
| WHATWG URL | [Relative URL string](https://url.spec.whatwg.org/#relative-url-string)与 [URL parsing](https://url.spec.whatwg.org/#url-parsing)：相对地址、路径和片段解析。 |
| W3C WAI | [Tables with Two Headers](https://www.w3.org/WAI/tutorials/tables/two-headers/)的行列表头；[Informative Images](https://www.w3.org/WAI/tutorials/images/informative/)的替代文本；[Labeling Controls](https://www.w3.org/WAI/tutorials/forms/labels/)与 [Grouping Controls](https://www.w3.org/WAI/tutorials/forms/grouping/)的标签及分组；[G1](https://www.w3.org/WAI/WCAG22/Techniques/general/G1#tests)的跳到主要内容；[Focus Order](https://www.w3.org/WAI/WCAG22/Understanding/focus-order.html)与 [Easy Checks](https://www.w3.org/WAI/test-evaluate/easy-checks/)的键盘顺序和初步检查范围；[WAI-ARIA 1.2 aria-current](https://www.w3.org/TR/wai-aria-1.2/#aria-current)与 [aria-describedby](https://www.w3.org/TR/wai-aria-1.2/#aria-describedby)的当前项目与描述关联。 |
| MDN | [Resolving relative references](https://developer.mozilla.org/en-US/docs/Web/API/URL_API/Resolving_relative_references)、[aria-current](https://developer.mozilla.org/en-US/docs/Web/Accessibility/ARIA/Reference/Attributes/aria-current)、[tabindex](https://developer.mozilla.org/en-US/docs/Web/HTML/Reference/Global_attributes/tabindex)：路径、当前页与聚焦；[Constraint validation](https://developer.mozilla.org/en-US/docs/Web/HTML/Guides/Constraint_validation)及 [maxlength](https://developer.mozilla.org/en-US/docs/Web/HTML/Reference/Attributes/maxlength)：原生约束、UTF-16 码元与服务端边界；[max-width](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/max-width)与 [:focus-visible](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Selectors/:focus-visible)：图片宽度和焦点样式。 |
| Chrome for Developers | [Network reference](https://developer.chrome.com/docs/devtools/network/reference#payload)的请求路径、状态及 Payload 查询字段查看。 |
| Python 3.12 | [http.server](https://docs.python.org/3.12/library/http.server.html#http.server.SimpleHTTPRequestHandler)的静态目录与请求处理；[urllib.parse](https://docs.python.org/3.12/library/urllib.parse.html#urllib.parse.parse_qsl)的查询解析、重复条目和空值；[html.escape](https://docs.python.org/3.12/library/html.html#html.escape)的文本转义。 |